Correcao
janela reduzida para 10s (sinais de 63s tem pouco ruido antes do evento)

Margem reduzida para 3s

Multiplas estações 

Amplitude zero -> investigação


In [1]:
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
warnings.filterwarnings('ignore')

PASTA_RAIZ    = r"C:\Users\alvaro.careli\Documents\TCC\data\scedc-pds"
PASTA_XML     = os.path.join(PASTA_RAIZ, "FDSNstationXML")
PASTA_PROJETO = r"C:\Users\alvaro.careli\Documents\TCC\Trabalho\artefacts"
INVENTARIO    = os.path.join(PASTA_PROJETO, "data", "inventario_dados.csv")
PASTA_WINDOWS = os.path.join(PASTA_PROJETO, "data", "windows")
os.makedirs(PASTA_WINDOWS, exist_ok=True)
os.makedirs(os.path.join(PASTA_PROJETO, "figures"), exist_ok=True)

# ── PARÂMETROS (ajustados para sinais curtos de 63s) ─────────────
CANAL_ALVO  = "BHZ"
SR_ALVO     = 40.0 
OUTPUT_RESP = "VEL"
PRE_FILT    = (0.5, 1.0, 18.0, 20.0)
WATER_LEVEL = 60
FREQ_MIN    = 0.5
FREQ_MAX    = 15.0

JANELA_SEG   = 10.0   # ← reduzido de 30s para 10s
SOBREPOSICAO = 0.5
MARGEM_SEG   = 3.0    # ← reduzido de 10s para 3s
N_AMOSTRAS   = int(JANELA_SEG * SR_ALVO)  # 400 amostras

print("✅ Configuração")
print(f"   Janela       : {JANELA_SEG}s = {N_AMOSTRAS} amostras")
print(f"   Sobreposição : {int(SOBREPOSICAO*100)}%")
print(f"   Margem       : {MARGEM_SEG}s antes do evento")
print(f"   Filtro       : {FREQ_MIN}–{FREQ_MAX} Hz")
print()
print("   MUDANÇA PRINCIPAL: usando TODAS as estações BHZ de cada arquivo")
print("   Isso multiplica o volume de dados ~50x")


✅ Configuração
   Janela       : 10.0s = 400 amostras
   Sobreposição : 50%
   Margem       : 3.0s antes do evento
   Filtro       : 0.5–15.0 Hz

   MUDANÇA PRINCIPAL: usando TODAS as estações BHZ de cada arquivo
   Isso multiplica o volume de dados ~50x


In [4]:
!pip install pandas

  Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl (9.9 MB)
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
 

In [5]:
import pandas as pd

df_inv = pd.read_csv(INVENTARIO)
df_uso = df_inv[
    (df_inv['xml_disponivel'] == True) &
    (df_inv['razao_var'] > 10)
].copy().reset_index(drop=True)

print(f"Arquivos para processar: {len(df_uso)}")
print(f"Duração média          : {df_uso['duracao_s'].mean():.0f}s")
print(f"t_evento médio         : {df_uso['t_evento_est'].mean():.1f}s")
print()
# Mostrar quantas janelas de ruído esperamos por arquivo com nova config
print("Estimativa de janelas de ruído por arquivo (nova config):")
passo = int(N_AMOSTRAS * (1 - SOBREPOSICAO))
for _, row in df_uso.head(5).iterrows():
    zona_ruido = max(0, row['t_evento_est'] - MARGEM_SEG)
    n_est = max(0, int((zona_ruido * SR_ALVO - N_AMOSTRAS) / passo) + 1)
    print(f"  {row['arquivo']}  t_evento={row['t_evento_est']}s  "
          f"zona_ruido={zona_ruido:.1f}s  ~{n_est} janelas ruído")


Arquivos para processar: 22
Duração média          : 64s
t_evento médio         : 22.3s

Estimativa de janelas de ruído por arquivo (nova config):
  37509232.ms  t_evento=27.0s  zona_ruido=24.0s  ~3 janelas ruído
  37509240.ms  t_evento=11.5s  zona_ruido=8.5s  ~1 janelas ruído
  37509256.ms  t_evento=37.5s  zona_ruido=34.5s  ~5 janelas ruído
  37509264.ms  t_evento=30.0s  zona_ruido=27.0s  ~4 janelas ruído
  37509272.ms  t_evento=25.0s  zona_ruido=22.0s  ~3 janelas ruído


In [3]:
!pip install obspy

In [ ]:
from obspy import read, read_inventory

def encontrar_xml(rede, estacao, pasta_xml):
    for root, _, files in os.walk(pasta_xml):
        for f in files:
            if f.endswith('.xml') and rede in f and estacao in f:
                return os.path.join(root, f)
    return None


def processar_trace(tr_input, caminho_xml):
    """
    Aplica o pipeline em um Trace já carregado (não em arquivo).
    Recebe um objeto obspy.Trace diretamente.
    """
    try:
        tr = tr_input.copy()

        if tr.stats.sampling_rate != SR_ALVO:
            tr.resample(SR_ALVO)

        tr.detrend('linear')
        tr.detrend('demean')
        tr.taper(max_percentage=0.05, type='cosine')

        inv = read_inventory(caminho_xml)
        tr.remove_response(inventory=inv, output=OUTPUT_RESP,
                           pre_filt=PRE_FILT, water_level=WATER_LEVEL)
        tr.filter('bandpass', freqmin=FREQ_MIN, freqmax=FREQ_MAX, zerophase=True)

        dados = tr.data.astype(np.float32)

        # Verificar se o sinal é válido (não zerado)
        if dados.std() < 1e-15:
            return None

        return dados
    except Exception:
        return None


def separar_janelas(sinal, sr, t_evento, janela_seg=JANELA_SEG,
                    sobreposicao=SOBREPOSICAO, margem_seg=MARGEM_SEG):
    n_win  = int(janela_seg * sr)
    passo  = int(n_win * (1 - sobreposicao))
    corte_ruido  = int((t_evento - margem_seg) * sr)
    corte_evento = int((t_evento - 2.0) * sr)  # 2s antes do pico

    ruido, evento = [], []
    for inicio in range(0, len(sinal) - n_win + 1, passo):
        fim    = inicio + n_win
        janela = sinal[inicio:fim].copy()
        std    = janela.std()
        if std > 1e-10:
            janela = (janela - janela.mean()) / std
        if   fim   <= corte_ruido  : ruido.append(janela)
        elif inicio >= corte_evento: evento.append(janela)

    r = np.array(ruido,  dtype=np.float32) if ruido  else np.empty((0,n_win), dtype=np.float32)
    e = np.array(evento, dtype=np.float32) if evento else np.empty((0,n_win), dtype=np.float32)
    return r, e


print("✅ Funções definidas")


ModuleNotFoundError: No module named 'pkg_resources'